# 08 — Scope Resolution

`HolonicDataset.resolve(predicate, from_holon, ...)` walks the
holarchy in BFS order and returns holons matching a predicate. Three
walk topologies:

- `"network"` (default) — outbound + inbound portals
- `"reverse-network"` — inbound portals only (debugging)
- `"containment"` — `cga:memberOf` chain

Two built-in predicate classes:

- `HasClassInInterior(class_iri)` — uses the 0.3.3 class inventory
  when present, falls back to direct interior query otherwise
- `CustomSPARQL(ask_template)` — escape hatch for any ASK-expressible
  condition

This notebook covers both, plus the ordering modes and limit clamps.


## Setup — a linear chain of holons


In [ ]:
from holonic import HolonicDataset, HasClassInInterior, CustomSPARQL

ds = HolonicDataset()

# A → B → C → D, each with a distinct class in its interior
for name, cls in [("a", "Alpha"), ("b", "Beta"), ("c", "Gamma"), ("d", "Delta")]:
    ds.add_holon(f"urn:holon:{name}", name.upper())
    ds.add_interior(f"urn:holon:{name}", f'''
        @prefix ex: <urn:ex:> .
        <urn:entity:{name}> a ex:{cls} .
    ''')

# Connect them sequentially
for src, tgt in [("a","b"), ("b","c"), ("c","d")]:
    ds.add_portal(
        f"urn:portal:{src}-{tgt}",
        source_iri=f"urn:holon:{src}",
        target_iri=f"urn:holon:{tgt}",
        construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
        label=f"{src}->{tgt}",
    )


## HasClassInInterior: find holons by interior type

From A, looking for a holon containing `ex:Gamma`. That's C, two hops
away via the portal chain.


In [ ]:
matches = ds.resolve(
    HasClassInInterior("urn:ex:Gamma"),
    from_holon="urn:holon:a",
    max_depth=3,
)
for m in matches:
    print(f"  {m.iri}  distance={m.distance}  evidence={m.evidence}")


## max_depth limits the walk

With `max_depth=1`, only A (self at distance 0) and B (1-hop) are
visited. C and D are out of range.


In [ ]:
matches_shallow = ds.resolve(
    HasClassInInterior("urn:ex:Delta"),
    from_holon="urn:holon:a",
    max_depth=1,
)
print(f"Delta at depth ≤1: {len(matches_shallow)}")

matches_deep = ds.resolve(
    HasClassInInterior("urn:ex:Delta"),
    from_holon="urn:holon:a",
    max_depth=5,
)
print(f"Delta at depth ≤5: {len(matches_deep)}  — {matches_deep[0].iri}")


## Reverse-network mode

Starting from D, "who depends on me?" Walk inbound portals only.


In [ ]:
matches = ds.resolve(
    HasClassInInterior("urn:ex:Alpha"),
    from_holon="urn:holon:d",
    order="reverse-network",
    max_depth=5,
)
print(f"A found via reverse-network: {matches[0].iri} (distance {matches[0].distance})")


## Containment mode: `cga:memberOf` walk

Build a small parent/child structure. Containment walks the
`cga:memberOf` chain in both directions (parent + descendants).


In [ ]:
ds.add_holon("urn:holon:parent", "Parent")
ds.add_holon("urn:holon:child", "Child", member_of="urn:holon:parent")
ds.add_interior("urn:holon:parent", '@prefix ex: <urn:ex:> . <urn:p> a ex:Parent .')
ds.add_interior("urn:holon:child", '@prefix ex: <urn:ex:> . <urn:c> a ex:Child .')

# From child, find parent's class via containment walk
matches = ds.resolve(
    HasClassInInterior("urn:ex:Parent"),
    from_holon="urn:holon:child",
    order="containment",
)
print(f"Parent found via containment: {matches[0].iri}")


## CustomSPARQL — the escape hatch

For predicates not covered by the built-in classes, supply an ASK
template. Placeholders `{holon_iri}` and `{registry_iri}` are
substituted via `str.replace` so normal SPARQL braces work without
escaping.

Below: find any holon with rdfs:label starting with "B".


In [ ]:
predicate = CustomSPARQL('''
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    ASK WHERE {
        GRAPH ?g {
            <{holon_iri}> rdfs:label ?label .
            FILTER(STRSTARTS(?label, "B"))
        }
    }
''')

matches = ds.resolve(predicate, from_holon="urn:holon:a", max_depth=5)
for m in matches:
    print(f"  {m.iri}  label starts with B")


## Star topology — all 1-hop neighbors ranked equally

BFS orders matches by distance first, then alphabetically by IRI
within a hop (for determinism).


In [ ]:
ds.add_holon("urn:holon:hub", "Hub")
ds.add_interior("urn:holon:hub", "@prefix ex: <urn:ex:> . <urn:h> a ex:Hub .")

for spoke in ("s1", "s2", "s3"):
    ds.add_holon(f"urn:holon:{spoke}", spoke.upper())
    ds.add_interior(f"urn:holon:{spoke}", f'@prefix ex: <urn:ex:> . <urn:{spoke}> a ex:Spoke .')
    ds.add_portal(
        f"urn:portal:hub-{spoke}",
        source_iri="urn:holon:hub",
        target_iri=f"urn:holon:{spoke}",
        construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
        label=f"hub→{spoke}",
    )

spokes = ds.resolve(HasClassInInterior("urn:ex:Spoke"), from_holon="urn:holon:hub")
for m in spokes:
    print(f"  {m.iri}  distance={m.distance}")


## limit clamps

`limit` clamps to `[1, 10_000]`. Below, only the first 2 spokes are
returned even though 3 exist.


In [ ]:
capped = ds.resolve(
    HasClassInInterior("urn:ex:Spoke"),
    from_holon="urn:holon:hub",
    limit=2,
)
print(f"capped count: {len(capped)}")


## Where to go next

- `09_console_views` — alternative "who's near this holon?" via
  `holon_neighborhood()` when you want the full subgraph rather
  than predicate-matched holons
- `10_projection_plugins` — scoped discovery paired with projection
  pipelines for "run X on every holon containing class Y" workflows
